<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session12_seq2seq_and_attention_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۲: مدل‌های Sequence-to-Sequence و سازوکار Attention

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session12_seq2seq_and_attention.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [1]:
import random
import time
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(0)
torch.manual_seed(0)
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


class CharTask:
    """Character vocabularies and data loaders for a (source text -> target text) task."""

    def __init__(self, pairs, n_train, n_val):
        self.src_itos = SPECIALS + sorted({c for s, _ in pairs for c in s})
        self.tgt_itos = SPECIALS + sorted({c for _, t in pairs for c in t})
        self.src_stoi = {c: i for i, c in enumerate(self.src_itos)}
        self.tgt_stoi = {c: i for i, c in enumerate(self.tgt_itos)}
        self.train, self.val, self.test = pairs[:n_train], pairs[n_train:n_train + n_val], pairs[n_train + n_val:]
        self.train_loader = self.loader(self.train, shuffle=True)

    def encode_src(self, s):
        return torch.tensor([self.src_stoi.get(c, UNK) for c in s] + [EOS])   # unknown characters -> <unk>

    def loader(self, pairs, batch_size=128, shuffle=False):
        data = [(self.encode_src(s), torch.tensor([SOS] + [self.tgt_stoi[c] for c in t] + [EOS])) for s, t in pairs]
        return DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

    def to_text(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            out.append(self.tgt_itos[i] if i >= len(SPECIALS) else '')
        return ''.join(out)


def collate(batch):
    src, tgt = zip(*batch)
    return pad_sequence(src, batch_first=True, padding_value=PAD), pad_sequence(tgt, batch_first=True, padding_value=PAD)


# the date task of the session notebook
FORMATS = [
    lambda d: d.strftime('%B %d, %Y'),
    lambda d: f'{d.day} {d.strftime("%B")} {d.year}',
    lambda d: f'{d.day} {d.strftime("%b")} {d.year}',
    lambda d: d.strftime('%A, %B %d, %Y'),
    lambda d: f'{d.strftime("%a")} {d.day} {d.strftime("%b")} {d.year}',
    lambda d: d.strftime('%d.%m.%Y'),
    lambda d: d.strftime('%Y/%m/%d'),
    lambda d: f'{d.strftime("%B")} {d.day} {d.year}',
    lambda d: f'the {d.day} of {d.strftime("%B")}, {d.year}',
]


def random_date():
    start = dt.date(1950, 1, 1).toordinal()
    return dt.date.fromordinal(random.randint(start, dt.date(2050, 12, 31).toordinal()))


def make_pairs(formats, n):
    out = []
    for _ in range(n):
        d = random_date()
        out.append((random.choice(formats)(d).lower(), d.isoformat()))
    return out


dates = CharTask(make_pairs(FORMATS, 30_000), 24_000, 3_000)


class Encoder(nn.Module):
    def __init__(self, vocab, emb=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.rnn = nn.GRU(emb, hidden, batch_first=True, bidirectional=True)
        self.bridge = nn.Linear(2 * hidden, hidden)

    def forward(self, src):
        outputs, h = self.rnn(self.embedding(src))
        return outputs, torch.tanh(self.bridge(torch.cat([h[0], h[1]], dim=1))).unsqueeze(0)


class AdditiveAttention(nn.Module):
    def __init__(self, dec_hidden, enc_hidden, attn_dim=128):
        super().__init__()
        self.W_s = nn.Linear(dec_hidden, attn_dim, bias=False)
        self.W_h = nn.Linear(enc_hidden, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, s, enc_outputs, mask):
        scores = self.v(torch.tanh(self.W_s(s).unsqueeze(1) + self.W_h(enc_outputs))).squeeze(-1)
        weights = torch.softmax(scores.masked_fill(~mask, float('-inf')), dim=1)
        return torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1), weights


class AttnDecoder(nn.Module):
    def __init__(self, vocab, emb=64, hidden=128, enc_hidden=256, attention=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.attention = attention or AdditiveAttention(hidden, enc_hidden)
        self.rnn = nn.GRU(emb + enc_hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden + enc_hidden, vocab)

    def step(self, token, h, enc_outputs, src_mask):
        """One decoding step. token: (B,) -> logits (B, V), new state, attention weights (B, S)"""
        context, w = self.attention(h[-1], enc_outputs, src_mask)
        out, h = self.rnn(torch.cat([self.embedding(token), context], dim=1).unsqueeze(1), h)
        return self.out(torch.cat([out.squeeze(1), context], dim=1)), h, w

    def forward(self, tgt_in, h, enc_outputs, src_mask, p_sample=0.0):
        """Teacher forcing; with probability p_sample the model's own previous prediction is fed instead."""
        logits, weights, token = [], [], tgt_in[:, 0]
        for t in range(tgt_in.shape[1]):
            if t > 0:
                token = tgt_in[:, t]
                if p_sample > 0:
                    use_pred = torch.rand(token.shape[0], device=token.device) < p_sample
                    token = torch.where(use_pred, logits[-1].argmax(-1), token)
            lg, h, w = self.step(token, h, enc_outputs, src_mask)
            logits.append(lg); weights.append(w)
        return torch.stack(logits, dim=1), h, torch.stack(weights, dim=1)


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder

    def forward(self, src, tgt_in, p_sample=0.0):
        enc_out, h = self.encoder(src)
        return self.decoder(tgt_in, h, enc_out, src != PAD, p_sample)[0]

    @torch.no_grad()
    def greedy_decode(self, src, max_len=12):
        enc_out, h = self.encoder(src)
        token = torch.full((src.shape[0],), SOS, dtype=torch.long, device=src.device)
        outputs = []
        for _ in range(max_len):
            logits, h, _ = self.decoder.step(token, h, enc_out, src != PAD)
            token = logits.argmax(-1)
            outputs.append(token)
        return torch.stack(outputs, dim=1)


@torch.no_grad()
def exact_match(model, task, pairs, max_len=12):
    model.eval()
    correct, i = 0, 0
    for src, _ in task.loader(pairs, batch_size=512):
        for row in model.greedy_decode(src.to(device), max_len).tolist():
            correct += task.to_text(row) == pairs[i][1]; i += 1
    return correct / len(pairs)


def train_seq2seq(model, task, epochs=5, lr=2e-3, p_schedule=None, max_len=12, verbose=True):
    """p_schedule(epoch) -> probability of feeding the model's own prediction (scheduled sampling)."""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        p = p_schedule(epoch) if p_schedule else 0.0
        for src, tgt in task.train_loader:
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src, tgt[:, :-1], p_sample=p)
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        history.append(exact_match(model, task, task.val, max_len))
        if verbose:
            print(f'epoch {epoch} | p_sample {p:.2f} | val exact match {history[-1]:.4f}')
    return history


def make_model(task, attention=None):
    return Seq2Seq(Encoder(len(task.src_itos)), AttnDecoder(len(task.tgt_itos), attention=attention))


torch.manual_seed(0)
attn_model = make_model(dates)
_ = train_seq2seq(attn_model, dates, epochs=5)

epoch 1 | p_sample 0.00 | val exact match 1.0000


epoch 2 | p_sample 0.00 | val exact match 1.0000


epoch 3 | p_sample 0.00 | val exact match 1.0000


epoch 4 | p_sample 0.00 | val exact match 1.0000


epoch 5 | p_sample 0.00 | val exact match 1.0000


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

**Attention ضرب نقطه‌ای (Luong / Dot-product)** را پیاده کنید: $e_{t,j} = \mathbf{s}_{t-1}^\top W \mathbf{h}_j$. نتیجه و سرعت را با Attention افزایشی مقایسه کنید.

</div>

In [2]:
class DotAttention(nn.Module):
    """Luong 'general' attention: e_tj = s^T W h_j."""

    def __init__(self, dec_hidden, enc_hidden):
        super().__init__()
        self.W = nn.Linear(enc_hidden, dec_hidden, bias=False)

    def forward(self, s, enc_outputs, mask):
        scores = torch.bmm(self.W(enc_outputs), s.unsqueeze(2)).squeeze(2)        # (B, S)
        weights = torch.softmax(scores.masked_fill(~mask, float('-inf')), dim=1)
        return torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1), weights


for name, attention in [('additive (Bahdanau)', None), ('multiplicative (Luong)', DotAttention(128, 256))]:
    torch.manual_seed(0)
    model = make_model(dates, attention)
    start = time.time()
    hist = train_seq2seq(model, dates, epochs=3, verbose=False)
    n_attn = sum(p.numel() for p in model.decoder.attention.parameters())
    print(f'{name:24}: attention params {n_attn:6,} | val exact match per epoch {np.round(hist, 4)} | {time.time() - start:.0f}s')

additive (Bahdanau)     : attention params 49,280 | val exact match per epoch [1. 1. 1.] | 86s


multiplicative (Luong)  : attention params 32,768 | val exact match per epoch [0.9963 1.     1.    ] | 319s


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در Attention ضرب نقطه‌ای، امتیاز فقط یک ضرب ماتریسی است: $W\mathbf{h}_j$ را می‌توان یک بار برای همهٔ مکان‌ها حساب کرد و سپس با یک `bmm` در $\mathbf{s}$ ضرب کرد. در Attention افزایشی باید برای **هر جفت** (گام Decoder، مکان ورودی) یک بردار `attn_dim` بعدی ساخت، از `tanh` گذراند و در $\mathbf{v}$ ضرب کرد. پس نسخهٔ ضربی پارامتر و محاسبات کمتری دارد. در این مسئله هر دو به دقت تقریباً کامل می‌رسند. در اجرای جدا، تفاوت زمانی کوچک است، چون حلقهٔ گام‌به‌گام Decoder بر هزینهٔ کل غالب است. (زمان‌های چاپ‌شده در این اجرا به‌خاطر اجرای همزمان برنامه‌های دیگر روی CPU قابل مقایسه نیستند.) ایدهٔ ضرب نقطه‌ای، با مقیاس $1/\sqrt{d}$، همان چیزی است که در Transformer (جلسهٔ ۱۳) استفاده می‌شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

چند قالب تاریخ جدید (مثلاً `3rd of march 2021` یا تاریخ با ماه‌های فارسی) به `FORMATS` اضافه کنید و ببینید مدل آموزش‌دیده چطور با قالب‌های ناآشنا برخورد می‌کند. سپس با داده جدید دوباره آموزش دهید.

</div>

In [3]:
def ordinal(n):
    suffix = 'th' if 11 <= n % 100 <= 13 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f'{n}{suffix}'


FA_MONTHS = ['ژانویه', 'فوریه', 'مارس', 'آوریل', 'مه', 'ژوئن', 'ژوئیه', 'اوت', 'سپتامبر', 'اکتبر', 'نوامبر', 'دسامبر']
NEW_FORMATS = [
    lambda d: f'{ordinal(d.day)} of {d.strftime("%B")} {d.year}',     # 3rd of march 2021
    lambda d: f'{d.day} {FA_MONTHS[d.month - 1]} {d.year}',           # 3 مارس 2021
]
random.seed(1)
unseen = make_pairs(NEW_FORMATS, 1000)

# 1) the model trained only on the old formats
for f_idx, name in enumerate(['ordinal English', 'Persian month names']):
    sub = make_pairs([NEW_FORMATS[f_idx]], 500)
    print(f'old model on {name:20}: exact match {exact_match(attn_model, dates, sub):.3f}')
    attn_model.eval()
    for s, t in sub[:3]:
        pred = dates.to_text(attn_model.greedy_decode(dates.encode_src(s).unsqueeze(0).to(device))[0].tolist())
        print(f'    {s:28} -> {pred}  (true {t})')

# 2) retrain on old + new formats (new vocabulary that includes the Persian letters)
random.seed(2)
dates2 = CharTask(make_pairs(FORMATS + NEW_FORMATS, 30_000), 24_000, 3_000)
torch.manual_seed(0)
model2 = make_model(dates2)
train_seq2seq(model2, dates2, epochs=5, verbose=False)
for f_idx, name in enumerate(['ordinal English', 'Persian month names']):
    print(f'retrained model on {name:20}: exact match {exact_match(model2, dates2, make_pairs([NEW_FORMATS[f_idx]], 500)):.3f}')

old model on ordinal English     : exact match 0.958
    3rd of december 1976         -> 1976-12-03  (true 1976-12-03)
    1st of august 1993           -> 1993-08-01  (true 1993-08-01)


    6th of june 1994             -> 1994-06-06  (true 1994-06-06)
old model on Persian month names : exact match 0.098
    16 مه 1997                   -> 1997-07-16  (true 1997-05-16)
    28 نوامبر 1968               -> 1968-07-28  (true 1968-11-28)
    20 فوریه 2033                -> 2033-07-20  (true 2033-02-20)


retrained model on ordinal English     : exact match 1.000
retrained model on Persian month names : exact match 1.000


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

- **قالب ترتیبی انگلیسی** (`3rd of march 2021`) به قالب‌های آموزشی نزدیک است: نام ماه و سال را قبلاً دیده است و فقط پسوندهای `st/nd/rd/th` جدیدند. مدل قدیمی معمولاً بخش زیادی از این نمونه‌ها را درست تبدیل می‌کند؛ Attention کمک می‌کند ارقام روز را پیدا کند و حروف اضافه را نادیده بگیرد.
- **نام ماه فارسی** برای مدل قدیمی کاملاً ناشناخته است: همهٔ حروف فارسی به `<unk>` تبدیل می‌شوند. روز و سال را می‌تواند درست بخواند، اما ماه را باید حدس بزند؛ پس دقت تطابق کامل نزدیک $1/12$ است.

بعد از آموزش دوباره با داده‌ای که قالب‌های جدید را هم دارد، هر دو قالب تقریباً کامل حل می‌شوند. **درس اصلی:** شبکهٔ عصبی بیرون از توزیع دادهٔ آموزشی (Out-of-Distribution) تضمینی ندارد. راه‌حل، پوشش دادن تنوع واقعی ورودی‌ها در دادهٔ آموزشی است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

**Scheduled Sampling** را پیاده کنید: در آموزش با احتمال $p$ (که به‌تدریج زیاد می‌شود) به‌جای Token درست، پیش‌بینی خود مدل را ورودی گام بعد کنید.

</div>

In [4]:
results = {}
for name, schedule in [('teacher forcing only', None),
                       ('scheduled sampling', lambda epoch: min(0.5, 0.1 * (epoch - 1)))]:   # 0, 0.1, ..., 0.4
    torch.manual_seed(0)
    model = make_model(dates)
    print(name)
    results[name] = train_seq2seq(model, dates, epochs=5, p_schedule=schedule)
    print(f'  test exact match {exact_match(model, dates, dates.test):.4f}')

teacher forcing only


epoch 1 | p_sample 0.00 | val exact match 1.0000


epoch 2 | p_sample 0.00 | val exact match 1.0000


epoch 3 | p_sample 0.00 | val exact match 1.0000


epoch 4 | p_sample 0.00 | val exact match 1.0000


epoch 5 | p_sample 0.00 | val exact match 1.0000


... (3 outputs left out)

epoch 3 | p_sample 0.20 | val exact match 1.0000


epoch 4 | p_sample 0.30 | val exact match 1.0000


epoch 5 | p_sample 0.40 | val exact match 1.0000


  test exact match 1.0000


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پیاده‌سازی در متد `forward` کلاس `AttnDecoder` (در سلول آماده‌سازی) است. در هر گام $t > 0$، برای هر نمونهٔ Batch جداگانه با احتمال $p$ به‌جای Token درست، پیش‌بینی گام قبل خود مدل (`argmax`) ورودی می‌شود. $p$ از صفر شروع و به‌تدریج زیاد می‌شود (Bengio et al., 2015). اول مدل با Teacher Forcing اصول را یاد می‌گیرد، سپس کم‌کم با خطاهای خودش روبه‌رو می‌شود تا **Exposure Bias** کم شود.

در این مسئلهٔ ساده، مدل با Teacher Forcing هم تقریباً کامل است و تفاوت دیده نمی‌شود. Scheduled Sampling در مسائلی سودمند است که خروجی بلند است و یک خطای کوچک در ابتدای تولید، بقیه را خراب می‌کند. نکته: چون نمونه‌گیری مشتق‌پذیر نیست، گرادیان از مسیر Token انتخاب‌شده عبور نمی‌کند؛ این یکی از محدودیت‌های نظری روش است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

Beam Search با اندازه پرتو ۳ را برای `attn_model` پیاده کنید.

</div>

In [5]:
@torch.no_grad()
def beam_search(model, task, text, beam_size=3, max_len=12):
    model.eval()
    src = task.encode_src(text).unsqueeze(0).to(device)
    enc_out, h = model.encoder(src)
    mask = src != PAD
    beams = [(0.0, [SOS], h, False)]                 # (log-probability, tokens, decoder state, finished)
    for _ in range(max_len):
        candidates = []
        for logp, tokens, h_b, done in beams:
            if done:
                candidates.append((logp, tokens, h_b, True))
                continue
            logits, h_new, _ = model.decoder.step(torch.tensor([tokens[-1]], device=device), h_b, enc_out, mask)
            log_probs = F.log_softmax(logits[0], dim=-1)
            top = log_probs.topk(beam_size)
            for lp, idx in zip(top.values.tolist(), top.indices.tolist()):
                candidates.append((logp + lp, tokens + [idx], h_new, idx == EOS))
        beams = sorted(candidates, key=lambda b: b[0], reverse=True)[:beam_size]
        if all(b[3] for b in beams):
            break
    return [(task.to_text(tokens[1:]), logp) for logp, tokens, _, _ in beams]


for text in ['march 3 2021', 'the 29 of february, 2024', 'thu 1 jan 1970', '31.12.1999']:
    print(text)
    for out, logp in beam_search(attn_model, dates, text):
        print(f'    {out:12} log p = {logp:8.4f}  (p = {np.exp(logp):.4f})')

march 3 2021
    2021-03-03   log p =  -0.0017  (p = 0.9983)
    2021-33-03   log p =  -7.2453  (p = 0.0007)
    2021-03-33   log p =  -7.9783  (p = 0.0003)
the 29 of february, 2024
    2024-02-29   log p =  -0.0007  (p = 0.9993)
    20242-02-29  log p =  -8.5812  (p = 0.0002)
    2024-02-09   log p =  -9.8033  (p = 0.0001)
thu 1 jan 1970
    1970-01-01   log p =  -0.0010  (p = 0.9990)
    1970-06-01   log p =  -8.6612  (p = 0.0002)
    1970-01-11   log p =  -8.9753  (p = 0.0001)
31.12.1999
    1999-12-31   log p =  -0.0011  (p = 0.9989)
    1999-12-319  log p =  -8.5295  (p = 0.0002)
    1999-12-311  log p =  -8.8703  (p = 0.0001)


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در هر گام، هر پرتو با ۳ Token محتمل بعدی گسترش می‌یابد و از همهٔ $3 \times 3$ نامزد، ۳ دنباله با بیشترین **مجموع لگاریتم احتمال** نگه داشته می‌شوند. دنباله‌هایی که به `<eos>` رسیده‌اند بدون تغییر در فهرست می‌مانند. برای این مسئلهٔ ساده، بهترین پرتو تقریباً همیشه همان خروجی Greedy است و احتمال آن نزدیک ۱ است. پرتوهای دوم و سوم احتمال بسیار کمی دارند و معمولاً در یک رقم تفاوت دارند. Beam Search در ترجمهٔ ماشینی و خلاصه‌سازی سود واقعی دارد، چون آنجا تصمیم Greedy در ابتدای جمله ممکن است ادامهٔ خوبی نداشته باشد. در عمل معمولاً امتیاز را بر طول دنباله نرمال می‌کنند تا دنباله‌های کوتاه ترجیح داده نشوند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

همین کد را برای یک مسئله «ترجمه» دیگر تغییر دهید: تبدیل اعداد نوشته‌شده با حروف انگلیسی به رقم (مثلاً `two hundred and five` به `205`).

</div>

In [6]:
ONES = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten', 'eleven', 'twelve',
        'thirteen', 'fourteen', 'fifteen', 'sixteen', 'seventeen', 'eighteen', 'nineteen']
TENS = ['', '', 'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty', 'ninety']


def below_1000(n):
    words = []
    if n >= 100:
        words += [ONES[n // 100], 'hundred']
        n %= 100
        if n:
            words.append('and')
    if n >= 20:
        words.append(TENS[n // 10] + ('-' + ONES[n % 10] if n % 10 else ''))
    elif n > 0 or not words:
        words.append(ONES[n])
    return ' '.join(words)


def number_to_words(n):
    if n < 1000:
        return below_1000(n)
    thousands, rest = divmod(n, 1000)
    words = below_1000(thousands) + ' thousand'
    if rest:
        words += (' and ' if rest < 100 else ' ') + below_1000(rest)
    return words


for n in [0, 7, 15, 40, 205, 999, 1001, 12345, 700000]:
    print(f'{n:>7} -> {number_to_words(n)}')

random.seed(3)
nums = [random.randint(0, 999_999) for _ in range(30_000)]
numbers = CharTask([(number_to_words(n), str(n)) for n in nums], 24_000, 3_000)
torch.manual_seed(0)
num_model = make_model(numbers)
train_seq2seq(num_model, numbers, epochs=8, max_len=8)
print(f'test exact match: {exact_match(num_model, numbers, numbers.test, max_len=8):.4f}')
num_model.eval()
for text in ['two hundred and five', 'nineteen thousand nine hundred and ninety-nine', 'five hundred thousand']:
    print(f'{text:48} -> {numbers.to_text(num_model.greedy_decode(numbers.encode_src(text).unsqueeze(0).to(device), 8)[0].tolist())}')

      0 -> zero
      7 -> seven
     15 -> fifteen
     40 -> forty
    205 -> two hundred and five
    999 -> nine hundred and ninety-nine
   1001 -> one thousand and one
  12345 -> twelve thousand three hundred and forty-five
 700000 -> seven hundred thousand


epoch 1 | p_sample 0.00 | val exact match 0.9830


epoch 2 | p_sample 0.00 | val exact match 0.9957


epoch 3 | p_sample 0.00 | val exact match 0.9987


epoch 4 | p_sample 0.00 | val exact match 0.9990


epoch 5 | p_sample 0.00 | val exact match 0.9993


epoch 6 | p_sample 0.00 | val exact match 0.9997


epoch 7 | p_sample 0.00 | val exact match 0.9997


epoch 8 | p_sample 0.00 | val exact match 0.9997


test exact match: 0.9997
two hundred and five                             -> 2055
nineteen thousand nine hundred and ninety-nine   -> 19999
five hundred thousand                            -> 500000


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تنها تغییر لازم، تولید داده است. کلاس `CharTask` واژگان کاراکتری را از روی خود داده می‌سازد و همان مدل و همان حلقهٔ آموزش بدون تغییر کار می‌کنند. حداکثر طول خروجی را ۸ گرفتیم (۶ رقم و `<eos>`). نکتهٔ جالب این مسئله، صفرهای «پنهان» است: در `two hundred and five` هیچ کلمه‌ای متناظر رقم صفر در `205` نیست و مدل باید از **ساختار** جمله بفهمد که دهگان خالی است. دقت تطابق کامل روی دادهٔ آزمون بسیار بالاست، اما خروجی بالا نشان می‌دهد مدل دقیقاً همین مثال را اشتباه کرد: `two hundred and five` را `2055` خواند. یعنی به‌جای پایان دادن دنباله بعد از `205`، یک رقم دیگر تولید کرد. نمونه‌های این‌چنینی در دادهٔ تصادفی ما نادرند (بیشتر اعداد شش‌رقمی‌اند)، پس دقت کلی بالا آن را پنهان می‌کند. **درس:** دقت کلی بالا جای تحلیل خطا روی موارد خاص را نمی‌گیرد. برای رفع آن، اعداد کوچک را در دادهٔ آموزشی بیشتر نمونه‌برداری کنید.

</div>